# CD-MSC — Prototype Inference (Exp A)

Evaluates whether **cosine similarity to per-species prototype embeddings** beats the standard softmax head on BAunseen — with no retraining.

**How it works:**
1. Load an existing trained checkpoint (use Exp 4 — the best we have)
2. Forward all training clips through the backbone to get 32-dim embeddings
3. Average per species → 9 prototype vectors
4. Classify test clips by cosine similarity to prototypes instead of softmax
5. Compare BAunseen: softmax vs prototype

**Why this might help:** Softmax is calibrated for D5. If C-DANN made the embedding somewhat domain-invariant, a D1 test clip's embedding may still be geometrically closer to the correct species prototype than to others — even if the softmax scores are miscalibrated.

**No GPU needed** — inference only on precomputed features. CPU runtime is fine.

In [ ]:
# Clone repo + install deps (no GPU required)
import os
if not os.path.exists('/content/CD-MSC'):
    !git clone https://github.com/saha23s/CD-MSC.git /content/CD-MSC
%cd /content/CD-MSC
!git checkout aaron/preprocessing
!git pull origin aaron/preprocessing
!pip install -q -r requirements.txt

In [ ]:
# Mount Drive + restore precomputed features
from google.colab import drive
drive.mount('/content/drive')

import shutil, pathlib
src = pathlib.Path('/content/drive/MyDrive/CD-MSC-feature')
dst = pathlib.Path('Development_data/feature')
dst.mkdir(parents=True, exist_ok=True)
if src.exists():
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    print(f'Features restored: {len(list(dst.glob("*.pkl")))} pkl files')
else:
    print('ERROR: MyDrive/CD-MSC-feature not found — run colab_quickstart.ipynb first')

In [ ]:
# List available experiment checkpoints on Drive so you can confirm the right one
DRIVE_OUTPUTS = pathlib.Path('/content/drive/MyDrive/CD-MSC-outputs')

print('Available experiments on Drive:')
for p in sorted(DRIVE_OUTPUTS.iterdir()):
    if p.is_dir():
        has_best  = (p / 'model' / 'model_best.pth').exists()
        has_final = (p / 'model' / 'model_final.pth').exists()
        has_cfg   = (p / 'resolved_config.json').exists()
        flags = ' '.join(filter(None, [
            'best' if has_best else '', 
            'final' if has_final else '',
            'cfg' if has_cfg else ''
        ]))
        print(f'  [{flags:20s}] {p.name}')

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Set this to the experiment you want to use as the source model.
# Exp 4 (C-DANN + balanced) is our best: BAunseen=0.2626

EXP_NAME = 'MTRCNN_seed42_B64_E100_earlystop_min10_pati5_dann0.3_cdann_balanced'

# Change to 'model_final.pth' to try the last epoch instead of the best checkpoint
CHECKPOINT_FILE = 'model_best.pth'
# ──────────────────────────────────────────────────────────────────────────────

EXP_DIR    = DRIVE_OUTPUTS / EXP_NAME
CHECKPOINT = EXP_DIR / 'model' / CHECKPOINT_FILE
CFG_PATH   = EXP_DIR / 'resolved_config.json'

assert EXP_DIR.exists(),   f'Experiment dir not found: {EXP_DIR}\nCheck the list above and update EXP_NAME.'
assert CHECKPOINT.exists(), f'Checkpoint not found: {CHECKPOINT}'
assert CFG_PATH.exists(),   f'resolved_config.json not found: {CFG_PATH}'

print(f'Experiment : {EXP_NAME}')
print(f'Checkpoint : {CHECKPOINT}')

In [ ]:
# Load config + build model
import json, sys, torch
sys.path.insert(0, '/content/CD-MSC')

from framework.dataset import MosquitoFeatureDataset, pad_collate_fn
from framework.engine import balanced_accuracy
from framework.metadata import SPECIES_NAMES, DOMAIN_NAMES
from framework.utilization import build_model, make_loader, split_feature_path, training_stats_path
import torch.nn.functional as F

with open(CFG_PATH) as f:
    config = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = build_model(config, device)
ckpt  = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

epoch  = ckpt.get('epoch', '?')
val_ba = ckpt.get('val_species_balanced_accuracy', float('nan'))
print(f'Checkpoint: epoch={epoch}, val_BA={val_ba:.4f}')

In [ ]:
# Helper: run one split through the model and collect embeddings + predictions
# Uses a forward hook on model.species_classifier to capture the 32-dim embedding
# (the embedding is the input to the species head — post-GELU, pre-classification).

def extract(dataloader):
    """Returns (embeddings [N,32], softmax_preds [N], species_labels [N], domain_names [N], file_ids [N])."""
    captured_emb, all_preds = [], []
    all_labels, all_domains, all_ids = [], [], []

    def _hook(module, inp, out):
        captured_emb.append(inp[0].detach().cpu())

    hook = model.species_classifier.register_forward_hook(_hook)
    with torch.no_grad():
        for batch in dataloader:
            features = batch['features'].to(device)
            lengths  = batch['lengths'].to(device)
            out = model(features, lengths)
            all_preds.append(out['species_logits'].argmax(dim=1).cpu())
            all_labels.extend(batch['species_labels'].tolist())
            all_domains.extend(batch['domain'])        # list of strings: 'D1', 'D5', etc.
            all_ids.extend(batch['file_id'])
    hook.remove()

    return (
        torch.cat(captured_emb, dim=0),   # [N, 32]
        torch.cat(all_preds,    dim=0),   # [N]
        all_labels,                        # list[int]
        all_domains,                       # list[str]
        all_ids,                           # list[str]
    )


def make_eval_dataset(split):
    return MosquitoFeatureDataset(
        feature_pickle_path=split_feature_path(config, split),
        feature_stats_path=training_stats_path(config),
        training=False,          # no augmentation, no random crop
        normalize_features=config['normalize_features'],
        cmn=config.get('cmn', False),
        use_delta=config.get('use_delta', False),
        use_approx_hpss=config.get('use_approx_hpss', False),
    )


bs = config.get('eval_batch_size', config['batch_size'])
nw = config['num_workers']

In [ ]:
# Extract training embeddings and compute per-species prototypes
train_ds     = make_eval_dataset('training')
train_loader = make_loader(train_ds, bs, False, nw, device, pad_collate_fn)

print(f'Extracting embeddings from {len(train_ds)} training clips...')
train_emb, _, train_labels_list, _, _ = extract(train_loader)
train_labels_t = torch.tensor(train_labels_list)

n_species  = len(SPECIES_NAMES)
prototypes = torch.zeros(n_species, 32)

print(f'\nPrototypes (mean embedding per species from {len(train_ds)} training clips):')
for s, name in enumerate(SPECIES_NAMES):
    mask = train_labels_t == s
    count = mask.sum().item()
    if count > 0:
        prototypes[s] = train_emb[mask].mean(dim=0)
    print(f'  {name:35s}: {count:6d} clips  norm={prototypes[s].norm():.3f}')

# L2-normalise for cosine similarity
prototypes_norm = F.normalize(prototypes, dim=1)  # [9, 32]
print(f'\nDone. Prototype matrix: {prototypes_norm.shape}')

In [ ]:
# Extract test embeddings, run softmax and prototype predictions
test_ds     = make_eval_dataset('test')
test_loader = make_loader(test_ds, bs, False, nw, device, pad_collate_fn)

print(f'Extracting embeddings from {len(test_ds)} test clips...')
test_emb, softmax_preds, test_labels_list, test_domains, test_ids = extract(test_loader)
test_labels_t = torch.tensor(test_labels_list)

# Prototype classification: cosine similarity to each centroid
test_emb_norm = F.normalize(test_emb, dim=1)      # [N, 32]
sim           = test_emb_norm @ prototypes_norm.T  # [N, 9]  (dot product of unit vectors = cosine sim)
proto_preds   = sim.argmax(dim=1)                  # [N]

print(f'Test clips: {len(test_labels_list)}')
print(f'Softmax predictions:   {softmax_preds.shape}')
print(f'Prototype predictions: {proto_preds.shape}')

In [ ]:
# Compute BAseen / BAunseen / DSG for both classifiers
# Reuse the same partition logic as evaluate.py
from evaluate import load_unseen_domain_by_species, append_official_metrics

unseen_by_species = load_unseen_domain_by_species(config)
print('Unseen domain per species:', unseen_by_species)


def make_rows(preds):
    return [
        {
            'file_id':                fid,
            'true_species_index':     lbl,
            'true_species_label':     SPECIES_NAMES[lbl],
            'true_domain_label':      dom,
            'predicted_species_index': int(pred),
        }
        for fid, lbl, dom, pred in zip(test_ids, test_labels_list, test_domains, preds.tolist())
    ]


def evaluate_preds(preds, label):
    metrics = {'species_balanced_accuracy': balanced_accuracy(preds, test_labels_t, n_species)}
    result  = append_official_metrics(metrics, make_rows(preds), unseen_by_species)
    m = result['metrics']
    print(f'\n{"="*52}')
    print(f'  {label}')
    print(f'{"="*52}')
    print(f'  BAseen   = {m["BA_seen"]:.4f}')
    print(f'  BAunseen = {m["BA_unseen"]:.4f}   ← primary metric')
    print(f'  DSG      = {m["DSG"]:.4f}')
    print(f'  Overall BA = {m["species_balanced_accuracy"]:.4f}')
    return m


softmax_m  = evaluate_preds(softmax_preds, 'Softmax head (standard)')
prototype_m = evaluate_preds(proto_preds,  'Prototype inference (cosine similarity)')

print(f'\n{"="*52}')
print( '  DELTA (Prototype − Softmax)')
print(f'{"="*52}')
print(f'  ΔBAseen   = {prototype_m["BA_seen"]   - softmax_m["BA_seen"]:+.4f}')
print(f'  ΔBAunseen = {prototype_m["BA_unseen"] - softmax_m["BA_unseen"]:+.4f}   ← what we care about')
print(f'  ΔDSG      = {prototype_m["DSG"]       - softmax_m["DSG"]:+.4f}')

In [ ]:
# Per-species recall breakdown
# Shows which species gained / lost recall from prototype vs softmax

def per_species_recall(preds):
    recalls = []
    for s in range(n_species):
        mask = test_labels_t == s
        if mask.sum() == 0:
            recalls.append(float('nan'))
        else:
            recalls.append((preds[mask] == s).float().mean().item())
    return recalls


soft_r  = per_species_recall(softmax_preds)
proto_r = per_species_recall(proto_preds)

print(f'\n{"Species":35s} {"Softmax":>10} {"Prototype":>10} {"Delta":>8}  Unseen domain')
print('-' * 82)
for s, name in enumerate(SPECIES_NAMES):
    unseen = unseen_by_species.get(name, '?')
    delta  = proto_r[s] - soft_r[s]
    flag   = '  ← GAIN' if delta > 0.02 else ('  ← LOSS' if delta < -0.02 else '')
    print(f'{name:35s} {soft_r[s]:10.3f} {proto_r[s]:10.3f} {delta:+8.3f}  {unseen}{flag}')

In [ ]:
# Save results to Drive
results = {
    'experiment':   EXP_NAME,
    'checkpoint':   CHECKPOINT_FILE,
    'softmax':      softmax_m,
    'prototype':    prototype_m,
    'delta_ba_unseen': prototype_m['BA_unseen'] - softmax_m['BA_unseen'],
}

out_path = DRIVE_OUTPUTS / EXP_NAME / 'prototype_eval.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to Drive: {out_path}')
print(f"\nFinal verdict: ΔBAunseen = {results['delta_ba_unseen']:+.4f}")